In [1]:
import matplotlib.style as mplstyle
mplstyle.use('fast')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from load_results import load, format
import matplotlib.pyplot as plt

import matplotlib.cm as cm
import numpy as np
import os
import pandas as pd
from sklearn.manifold import MDS
from scipy.linalg import orthogonal_procrustes
import math
import regex as re
import numpy as np
import numpy as np
import numpy as np
from adjustText import adjust_text
import pandas as pd
from load_results import format
df = load()
df = format(df)

In [2]:
from plotting_config import scores_to_plot

In [3]:

def rgba_to_hex(rgba):
    return '#{:02x}{:02x}{:02x}'.format(
        int(rgba[0]*255),
        int(rgba[1]*255),
        int(rgba[2]*255)
    )
from load_results import abbrev_map_strategy
def plot_mds_and_heatmaps(df, row_cols, col_col=None, save_dir="figures", main_fontsize=7, filter_fn=lambda df: df, display=True):
    os.makedirs(save_dir, exist_ok=True)
    for group_vals, df_setting in filter_fn(df).groupby(row_cols):
        if isinstance(group_vals, tuple):
            group_dict = dict(zip(row_cols, group_vals))
        else:
            group_dict = {row_cols[0]: group_vals}
        score_name = group_dict.get("score_name")
        k = group_dict.get("k")
        if score_name not in scores_to_plot:
            continue
        nodes = sorted(pd.unique(df_setting[["Rep_A", "Rep_B"]].values.ravel()))
        
        idx = {n: i for i, n in enumerate(nodes)}
        n = len(nodes)
        cmap = plt.get_cmap("tab20", n)
        

        if col_col is None:
            experiments = ["all"]
            grouped_experiments = [("all", df_setting)]
        else:
            experiments = df_setting[col_col].dropna().unique()
            grouped_experiments = list(df_setting.groupby(col_col))
            
            
        

        n_exp = len(experiments)
        max_cols = 4
        ncols = min(max_cols, n_exp)
        nrows = math.ceil(n_exp / ncols)
        square_size = 3
        
        filename_base = "_".join([col_col if col_col else "all"] + [f"{k}={v}" for k, v in group_dict.items()])
        filename_base = re.sub(r"[#$%&{}^~\\]", "", filename_base)
        filename_base = re.sub(r"[\s:/<>?|]+", "_", filename_base).lower()
        print(filename_base)
        
        for exp, df_exp in grouped_experiments:
            fig, ax = plt.subplots(
                figsize=(square_size, square_size),
            )

            df_tmp = df_exp.copy()
            df_tmp[["Rep_A", "Rep_B"]] = np.sort(df_tmp[["Rep_A", "Rep_B"]], axis=1)
            df_agg = df_tmp.groupby(["Rep_A", "Rep_B"], as_index=False)["score"].mean()

            adj = np.full((n, n), np.nan)
            np.fill_diagonal(adj, 1.0)

            for _, r in df_agg.iterrows():
                i, j = idx[r["Rep_A"]], idx[r["Rep_B"]]
                adj[i, j] = adj[j, i] = r["score"]

            nodes = [re.sub("|".join(map(re.escape, abbrev_map_strategy.values())), "$", node) for node in nodes]
            # print(nodes)
            colors = {node: cmap(i) for i, node in enumerate(nodes)}
            heatmap_data = pd.DataFrame(adj, index=nodes, columns=nodes)

            def fmt_custom(x):
                x = round(x, 2)

                if x == 0:
                    return ".00"

                if x == int(x):
                    return str(int(x))

                s = f"{x:.2f}"

                if s.startswith("-0"):
                    return "-" + s[2:]   # -0.50 → -.50
                if s.startswith("0"):
                    return s[1:]         # 0.50 → .50

                return s
            sns.heatmap(
                heatmap_data,
                annot=True,
                cmap="viridis",
                vmin=0, vmax=1,
                square=True,
                cbar=False,
                ax=ax,
                annot_kws={"size": main_fontsize}
            )

            for t in ax.texts:
                val = float(t.get_text())
                t.set_text(fmt_custom(val))

            # color labels
            for label in ax.get_xticklabels():
                node = label.get_text()
                label.set_color(rgba_to_hex(colors[node]))

            for label in ax.get_yticklabels():
                node = label.get_text()
                label.set_color(rgba_to_hex(colors[node]))

            if col_col:
                ax.text(
                    1.02, 0.5,                      # just outside the right edge, centered vertically
                    exp.replace("_", ""),
                    transform=ax.transAxes,
                    rotation=-90,                   # vertical title
                    va="center",
                    ha="left",
                    fontsize=main_fontsize 
                )

            plt.tight_layout()

            exp_safe = re.sub(r"[\\s:/<>?|]+", "_", str(exp)).lower()
            file_name = f"{filename_base}_{exp_safe}_heatmap.pdf"
            print(os.path.join(".",save_dir, file_name))
            plt.savefig(
                os.path.join(save_dir, file_name),
                format="pdf",
                bbox_inches="tight",
                pad_inches=0
            )
            if display:
                plt.show()
                break
            plt.close(fig)

In [4]:
for selection_strategy in ["most_influential", "least_influential", "most_harmful", "most_helpful"]:
    print(selection_strategy)
    plot_mds_and_heatmaps(
        df,
        row_cols=["score_name", "k", "normalize"],
        col_col=None,
        save_dir=os.path.join("figures/heatmaps/",selection_strategy),
        filter_fn=lambda df: df[
            df["selection_strategy_name_a"].isin([selection_strategy, "-"])
            & df["selection_strategy_name_b"].isin([selection_strategy, "-"])
        ],
        display=False
    )
    # break

most_influential
all_score_name=cka_k=-1_normalize=false


./figures/heatmaps/most_influential/all_score_name=cka_k=-1_normalize=false_all_heatmap.pdf


all_score_name=cknna_k=25_normalize=false
./figures/heatmaps/most_influential/all_score_name=cknna_k=25_normalize=false_all_heatmap.pdf


all_score_name=kendall_k=-1_normalize=false
./figures/heatmaps/most_influential/all_score_name=kendall_k=-1_normalize=false_all_heatmap.pdf


all_score_name=pw_acc_k=-1_normalize=false
./figures/heatmaps/most_influential/all_score_name=pw_acc_k=-1_normalize=false_all_heatmap.pdf


all_score_name=rbo_k=-1_normalize=false
./figures/heatmaps/most_influential/all_score_name=rbo_k=-1_normalize=false_all_heatmap.pdf


... (19 outputs left out)

all_score_name=kendall_k=-1_normalize=false
./figures/heatmaps/most_helpful/all_score_name=kendall_k=-1_normalize=false_all_heatmap.pdf


all_score_name=pw_acc_k=-1_normalize=false
./figures/heatmaps/most_helpful/all_score_name=pw_acc_k=-1_normalize=false_all_heatmap.pdf


all_score_name=rbo_k=-1_normalize=false
./figures/heatmaps/most_helpful/all_score_name=rbo_k=-1_normalize=false_all_heatmap.pdf


all_score_name=spearman_rho_k=-1_normalize=false
./figures/heatmaps/most_helpful/all_score_name=spearman_rho_k=-1_normalize=false_all_heatmap.pdf


In [5]:
for selection_strategy in ["most_influential", "least_influential", "most_harmful", "most_helpful"]:
    print(selection_strategy)
    plot_mds_and_heatmaps(
        df,
        row_cols=["score_name", "k", "normalize"],
        col_col="Experiment",
        save_dir=os.path.join("figures/heatmaps/",selection_strategy),
        filter_fn=lambda df: df[
            df["selection_strategy_name_a"].isin([selection_strategy, "-"])
            & df["selection_strategy_name_b"].isin([selection_strategy, "-"])
        ],
        display=False
    )

most_influential


experiment_score_name=cka_k=-1_normalize=false
./figures/heatmaps/most_influential/experiment_score_name=cka_k=-1_normalize=false_code_heatmap.pdf


./figures/heatmaps/most_influential/experiment_score_name=cka_k=-1_normalize=false_ftrace_heatmap.pdf


./figures/heatmaps/most_influential/experiment_score_name=cka_k=-1_normalize=false_math_heatmap.pdf


./figures/heatmaps/most_influential/experiment_score_name=cka_k=-1_normalize=false__ft_heatmap.pdf


./figures/heatmaps/most_influential/experiment_score_name=cka_k=-1_normalize=false__quad_v2_heatmap.pdf


... (115 outputs left out)

./figures/heatmaps/most_helpful/experiment_score_name=spearman_rho_k=-1_normalize=false_ftrace_heatmap.pdf


./figures/heatmaps/most_helpful/experiment_score_name=spearman_rho_k=-1_normalize=false_math_heatmap.pdf


./figures/heatmaps/most_helpful/experiment_score_name=spearman_rho_k=-1_normalize=false__ft_heatmap.pdf


./figures/heatmaps/most_helpful/experiment_score_name=spearman_rho_k=-1_normalize=false__quad_v2_heatmap.pdf


In [6]:
for selection_strategy in ["most_influential", "least_influential", "most_harmful", "most_helpful"]:
    print(selection_strategy)
    plot_mds_and_heatmaps(
        df,
        row_cols=["score_name", "k", "normalize"],
        col_col="base_model",
        save_dir=os.path.join("figures/heatmaps",selection_strategy),
        filter_fn=lambda df: df[
            df["selection_strategy_name_a"].isin([selection_strategy, "-"])
            & df["selection_strategy_name_b"].isin([selection_strategy, "-"])
        ],
        display=False
    )
    

most_influential
base_model_score_name=cka_k=-1_normalize=false


./figures/heatmaps/most_influential/base_model_score_name=cka_k=-1_normalize=false_llama-3.2-1b_heatmap.pdf


./figures/heatmaps/most_influential/base_model_score_name=cka_k=-1_normalize=false_olmo-2-0425-1b_heatmap.pdf


./figures/heatmaps/most_influential/base_model_score_name=cka_k=-1_normalize=false_olmo-3-1025-7b_heatmap.pdf


./figures/heatmaps/most_influential/base_model_score_name=cka_k=-1_normalize=false_qwen3-0.6b-ba_e_heatmap.pdf


./figures/heatmaps/most_influential/base_model_score_name=cka_k=-1_normalize=false_qwen3-1.7b-ba_e_heatmap.pdf


... (160 outputs left out)

./figures/heatmaps/most_helpful/base_model_score_name=spearman_rho_k=-1_normalize=false_qwen3-0.6b-ba_e_heatmap.pdf


./figures/heatmaps/most_helpful/base_model_score_name=spearman_rho_k=-1_normalize=false_qwen3-1.7b-ba_e_heatmap.pdf


./figures/heatmaps/most_helpful/base_model_score_name=spearman_rho_k=-1_normalize=false_qwen3-4b-ba_e_heatmap.pdf


./figures/heatmaps/most_helpful/base_model_score_name=spearman_rho_k=-1_normalize=false_qwen3-8b-ba_e_heatmap.pdf
